# Hypothesis Testing: 30-Day Readmission

## Objective
This notebook validates four project-specific hypotheses about factors associated with **30-day hospital readmission** using the cleaned Diabetes 130-US Hospitals dataset. The goal is not to jump directly to testing, but to first check whether the statistical assumptions and data conditions support each hypothesis test.

## Why this notebook matters
EDA showed that prior utilisation is the strongest signal, while admission/discharge patterns, clinical complexity, and medication management may also influence readmission risk. This notebook converts those patterns into formal statistical evidence.

## Important data validity note
The data dictionary states that patients with certain discharge disposition codes representing expired or hospice outcomes cannot be readmitted and are often removed during preprocessing. The current cleaned dataset keeps those records, so this notebook explicitly checks for impossible-readmission discharge groups before proceeding.

## Planned hypotheses
1. **Prior inpatient utilisation hypothesis**: patients with more prior inpatient visits have higher 30-day readmission risk.
2. **Length of stay hypothesis**: patients with longer hospital stays have higher 30-day readmission risk.
3. **Discharge disposition hypothesis**: discharge disposition is associated with 30-day readmission.
4. **Medication change hypothesis**: medication change status (`change`) is associated with 30-day readmission.

## Test selection logic
- Continuous/count variables vs binary target: compare the two target groups. Because healthcare utilisation variables are usually skewed, we formally test normality and then prefer a non-parametric Mann-Whitney U test when normality is not supported.
- Categorical variables vs binary target: use chi-square test of independence, and check expected cell counts before trusting the result.
- Effect size is reported with every hypothesis because large samples can make tiny differences statistically significant.


In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu, chi2_contingency, shapiro

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_PATHS = [
    Path('cleaned_data.csv'),
    Path('data/processed/cleaned_data.csv'),
    Path('../data/processed/cleaned_data.csv'),
]

for path in DATA_PATHS:
    if path.exists():
        df = pd.read_csv(path)
        data_path_used = path
        break
else:
    raise FileNotFoundError('cleaned_data.csv not found in expected locations.')

# Rename columns to match the expected names in the notebook
df = df.rename(columns={
    'time_in_hospital': 'timeinhospital',
    'number_inpatient': 'numberinpatient',
    'discharge_disposition': 'dischargedisposition',
    'readmitted_binary': 'readmittedbinary'
})

print(f'Dataset loaded from: {data_path_used}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Positive class (readmitted within 30 days): {df["readmittedbinary"].sum():,} ({df["readmittedbinary"].mean()*100:.2f}%)')
df.head()


Dataset loaded from: ..\data\processed\cleaned_data.csv
Shape: 101,766 rows x 47 columns
Positive class (readmitted within 30 days): 11,357 (11.16%)


,race,gender,timeinhospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,numberinpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmittedbinary,age_numeric,admission_type,dischargedisposition,admission_source
0,Caucasian,Female,1,Unknown,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,Unknown,Unknown,1,Unknown,Unknown,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0,5.0,NaN,Not Mapped,Physician Referral
1,Caucasian,Female,3,Unknown,Unknown,59,0,18,0,0,0,276,250.01,255,9,Unknown,Unknown,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0,15.0,Emergency,Discharged to home,Emergency Room
2,AfricanAmerican,Female,2,Unknown,Unknown,11,5,13,2,0,1,648,250,V27,6,Unknown,Unknown,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0,25.0,Emergency,Discharged to home,Emergency Room
3,Caucasian,Male,2,Unknown,Unknown,44,1,16,0,0,0,8,250.43,403,7,Unknown,Unknown,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0,35.0,Emergency,Discharged to home,Emergency Room
4,Caucasian,Male,1,Unknown,Unknown,51,0,8,0,0,0,197,157,250,5,Unknown,Unknown,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0,45.0,Emergency,Discharged to home,Emergency Room


## Step 1: Condition checks before hypothesis testing
We first verify data quality, target balance, column availability, and whether the dataset contains discharge groups that make readmission logically impossible. This prevents invalid interpretation later.


In [4]:
required_columns = [
    'readmittedbinary', 'numberinpatient', 'timeinhospital',
    'dischargedisposition', 'change'
]

missing_columns = [c for c in required_columns if c not in df.columns]
print('Missing required columns:', missing_columns if missing_columns else 'None')
print('Total null values in dataset:', int(df.isna().sum().sum()))

impossible_keywords = ['Expired', 'Hospice', 'hospice', 'expired']
impossible_mask = df['dischargedisposition'].astype(str).str.contains('|'.join(impossible_keywords), case=False, na=False)
impossible_rows = df.loc[impossible_mask, ['dischargedisposition', 'readmittedbinary']]

print(f'Rows in logically impossible discharge groups (expired/hospice): {len(impossible_rows):,}')
if len(impossible_rows) > 0:
    print('Readmission rate inside impossible discharge groups:', round(impossible_rows['readmittedbinary'].mean()*100, 4), '%')
    display(impossible_rows['dischargedisposition'].value_counts().rename_axis('dischargedisposition').reset_index(name='count'))
else:
    print('No expired/hospice discharge groups found in current cleaned dataset labels.')


Missing required columns: None
Total null values in dataset: 15763
Rows in logically impossible discharge groups (expired/hospice): 2,423
Readmission rate inside impossible discharge groups: 1.7747 %


,dischargedisposition,count
0,Expired,1642
1,Hospice / home,399
2,Hospice / medical facility,372
3,"Expired at home. Medicaid only, hospice.",8
4,"Expired in a medical facility. Medicaid only, ...",2


### Decision on analysis sample
If impossible-readmission discharge groups exist, we exclude them for hypothesis testing to keep the analysis logically consistent with the target definition. This creates an analysis-ready sample for inferential testing.


In [5]:
analysis_df = df.loc[~impossible_mask].copy()
print(f'Analysis sample shape: {analysis_df.shape[0]:,} rows x {analysis_df.shape[1]} columns')
print(f'Excluded rows: {df.shape[0] - analysis_df.shape[0]:,}')
print(f'Analysis sample readmission rate: {analysis_df["readmittedbinary"].mean()*100:.2f}%')


Analysis sample shape: 99,343 rows x 47 columns
Excluded rows: 2,423
Analysis sample readmission rate: 11.39%


## Helper functions
These functions automate condition checks, test selection, effect-size calculation, and interpretation so that each hypothesis is evaluated consistently.


In [6]:
def normality_check(series, max_n=5000, random_state=42):
    s = pd.Series(series).dropna()
    if len(s) < 3:
        return {'n_tested': len(s), 'p_value': np.nan, 'normal': False}
    sample = s.sample(min(len(s), max_n), random_state=random_state) if len(s) > max_n else s
    stat, p = shapiro(sample)
    return {'n_tested': len(sample), 'p_value': p, 'normal': p > 0.05}

def cliffs_delta(x, y):
    x = np.asarray(pd.Series(x).dropna())
    y = np.asarray(pd.Series(y).dropna())
    if len(x) == 0 or len(y) == 0:
        return np.nan
    greater = sum((xi > y).sum() for xi in x)
    lower = sum((xi < y).sum() for xi in x)
    return (greater - lower) / (len(x) * len(y))

def cohen_h_from_props(p1, p2):
    return 2*np.arcsin(np.sqrt(p1)) - 2*np.arcsin(np.sqrt(p2))

def cramers_v_from_table(table):
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.to_numpy().sum()
    r, c = table.shape
    return np.sqrt(chi2 / (n * max(1, min(r-1, c-1))))

def effect_label(value, kind):
    v = abs(value)
    if pd.isna(v):
        return 'NA'
    if kind == 'cliffs_delta':
        if v < 0.147: return 'negligible'
        if v < 0.33: return 'small'
        if v < 0.474: return 'medium'
        return 'large'
    if kind == 'cramers_v':
        if v < 0.10: return 'negligible'
        if v < 0.30: return 'small'
        if v < 0.50: return 'medium'
        return 'large'
    if kind == 'cohen_h':
        if v < 0.20: return 'small'
        if v < 0.50: return 'medium'
        if v < 0.80: return 'large'
        return 'very large'
    return 'NA'


## Hypothesis 1: Prior inpatient utilisation
### Research question
Do patients who were readmitted within 30 days have higher prior inpatient utilisation than those who were not?

### Statistical framing
- **H0**: The distribution of `numberinpatient` is the same in both readmission groups.
- **H1**: The distribution of `numberinpatient` is higher among patients readmitted within 30 days.

### Condition checks
1. Outcome has exactly two groups.
2. Predictor is numeric/count.
3. Group sizes are adequate.
4. Normality is checked in both groups; if violated, use Mann-Whitney U.


In [9]:
var = 'numberinpatient'
g0 = analysis_df.loc[analysis_df['readmittedbinary'] == 0, var]
g1 = analysis_df.loc[analysis_df['readmittedbinary'] == 1, var]

norm0 = normality_check(g0)
norm1 = normality_check(g1)

print('Group sizes:', {'Not readmitted': len(g0), 'Readmitted 30d': len(g1)})
print('Normality check (Shapiro sample-based):')
print({'Not readmitted': norm0, 'Readmitted 30d': norm1})

u_stat, p_value = mannwhitneyu(g1, g0, alternative='two-sided')
delta = cliffs_delta(g1, g0)

h1_result = {
    'Hypothesis': 'H1 Prior inpatient utilisation',
    'Variable': var,
    'Test': 'Mann-Whitney U',
    'Group0 median': g0.median(),
    'Group1 median': g1.median(),
    'Group0 mean': g0.mean(),
    'Group1 mean': g1.mean(),
    'Statistic': u_stat,
    'p_value': p_value,
    'Effect_size': delta,
    'Effect_type': "Cliff's delta",
    'Effect_magnitude': effect_label(delta, 'cliffs_delta'),
    'Decision': 'Reject H0' if p_value < 0.05 else 'Fail to reject H0'
}
pd.DataFrame([h1_result]).round(4)

Group sizes: {'Not readmitted': 88029, 'Readmitted 30d': 11314}
Normality check (Shapiro sample-based):
{'Not readmitted': {'n_tested': 5000, 'p_value': np.float64(3.0971260486438075e-78), 'normal': np.False_}, 'Readmitted 30d': {'n_tested': 5000, 'p_value': np.float64(2.2326213810769003e-72), 'normal': np.False_}}


,Hypothesis,Variable,Test,Group0 median,Group1 median,Group0 mean,Group1 mean,Statistic,p_value,Effect_size,Effect_type,Effect_magnitude,Decision
0,H1 Prior inpatient utilisation,numberinpatient,Mann-Whitney U,0.0,0.0,0.5549,1.2227,604815339.5,0.0,0.2145,Cliff's delta,small,Reject H0


## Hypothesis 2: Length of stay
### Research question
Do patients readmitted within 30 days have longer hospital stays than those not readmitted within 30 days?

### Statistical framing
- **H0**: The distribution of `timeinhospital` is the same across readmission groups.
- **H1**: Readmitted patients have longer hospital stays.

### Condition checks
The same checks are repeated here because test validity must be assessed per variable, not assumed from a previous hypothesis.


In [12]:
var = 'timeinhospital'
g0 = analysis_df.loc[analysis_df['readmittedbinary'] == 0, var]
g1 = analysis_df.loc[analysis_df['readmittedbinary'] == 1, var]

norm0 = normality_check(g0)
norm1 = normality_check(g1)

print('Group sizes:', {'Not readmitted': len(g0), 'Readmitted 30d': len(g1)})
print('Normality check (Shapiro sample-based):')
print({'Not readmitted': norm0, 'Readmitted 30d': norm1})

u_stat, p_value = mannwhitneyu(g1, g0, alternative='two-sided')
delta = cliffs_delta(g1, g0)

h2_result = {
    'Hypothesis': 'H2 Length of stay',
    'Variable': var,
    'Test': 'Mann-Whitney U',
    'Group0 median': g0.median(),
    'Group1 median': g1.median(),
    'Group0 mean': g0.mean(),
    'Group1 mean': g1.mean(),
    'Statistic': u_stat,
    'p_value': p_value,
    'Effect_size': delta,
    'Effect_type': "Cliff's delta",
    'Effect_magnitude': effect_label(delta, 'cliffs_delta'),
    'Decision': 'Reject H0' if p_value < 0.05 else 'Fail to reject H0'
}
pd.DataFrame([h2_result]).round(4)

Group sizes: {'Not readmitted': 88029, 'Readmitted 30d': 11314}
Normality check (Shapiro sample-based):
{'Not readmitted': {'n_tested': 5000, 'p_value': np.float64(4.55434147571632e-51), 'normal': np.False_}, 'Readmitted 30d': {'n_tested': 5000, 'p_value': np.float64(1.5519604696463032e-47), 'normal': np.False_}}


,Hypothesis,Variable,Test,Group0 median,Group1 median,Group0 mean,Group1 mean,Statistic,p_value,Effect_size,Effect_type,Effect_magnitude,Decision
0,H2 Length of stay,timeinhospital,Mann-Whitney U,4.0,4.0,4.3294,4.7675,545488921.5,0.0,0.0954,Cliff's delta,negligible,Reject H0


## Hypothesis 3: Discharge disposition
### Research question
Is discharge disposition associated with 30-day readmission?

### Statistical framing
- **H0**: `dischargedisposition` and `readmittedbinary` are independent.
- **H1**: `dischargedisposition` and `readmittedbinary` are associated.

### Condition checks
1. Both variables are categorical.
2. Contingency table has adequate cell counts.
3. If too many expected counts are below 5, interpretation becomes weaker and grouping may be needed.


In [14]:
var = 'dischargedisposition'
top_levels = analysis_df[var].value_counts().head(12).index
temp = analysis_df.copy()
temp[var] = np.where(temp[var].isin(top_levels), temp[var], 'Other')
table = pd.crosstab(temp[var], temp['readmittedbinary'])
chi2, p_value, dof, expected = chi2_contingency(table)
expected_lt5 = (expected < 5).sum()
cramers_v = cramers_v_from_table(table)

print('Contingency table shape:', table.shape)
print('Cells with expected count < 5:', int(expected_lt5))
display(table.head(15))

h3_result = {
    'Hypothesis': 'H3 Discharge disposition',
    'Variable': var,
    'Test': 'Chi-square test of independence',
    'Statistic': chi2,
    'Degrees_of_freedom': dof,
    'p_value': p_value,
    'Effect_size': cramers_v,
    'Effect_type': "Cramer's V",
    'Effect_magnitude': effect_label(cramers_v, 'cramers_v'),
    'Expected_cells_lt5': int(expected_lt5),
    'Decision': 'Reject H0' if p_value < 0.05 else 'Fail to reject H0'
}
pd.DataFrame([h3_result]).round(4)

Contingency table shape: (13, 2)
Cells with expected count < 5: 0


readmittedbinary,0,1
dischargedisposition,,
Discharged to home,54632,5602
Discharged/transferred to ICF,711,104
Discharged/transferred to SNF,11908,2046
Discharged/transferred to a long term care hospital.,382,30
Discharged/transferred to another rehab fac including rehab units of a hospital .,1441,552
Discharged/transferred to another short term hospital,1786,342
Discharged/transferred to another type of inpatient care institution,937,247
Discharged/transferred to home under care of Home IV provider,93,15
Discharged/transferred to home with home health service,11264,1638


,Hypothesis,Variable,Test,Statistic,Degrees_of_freedom,p_value,Effect_size,Effect_type,Effect_magnitude,Expected_cells_lt5,Decision
0,H3 Discharge disposition,dischargedisposition,Chi-square test of independence,1225.2637,12,0.0,0.1111,Cramer's V,small,0,Reject H0


## Hypothesis 4: Medication change
### Research question
Is medication change status (`change`) associated with 30-day readmission?

### Statistical framing
- **H0**: `change` and `readmittedbinary` are independent.
- **H1**: medication change is associated with 30-day readmission.

### Condition checks
Because `change` has a very small number of categories, chi-square assumptions are typically easier to satisfy. We still verify the expected counts before interpreting the result.


In [15]:
var = 'change'
table = pd.crosstab(analysis_df[var], analysis_df['readmittedbinary'])
chi2, p_value, dof, expected = chi2_contingency(table)
expected_lt5 = (expected < 5).sum()
cramers_v = cramers_v_from_table(table)

prop_change = table.div(table.sum(axis=1), axis=0)
p_ch = prop_change.loc['Ch', 1] if 'Ch' in prop_change.index else np.nan
p_no = prop_change.loc['No', 1] if 'No' in prop_change.index else np.nan
cohen_h = cohen_h_from_props(p_ch, p_no) if pd.notna(p_ch) and pd.notna(p_no) else np.nan

print('Contingency table:')
display(table)
print('Cells with expected count < 5:', int(expected_lt5))

h4_result = {
    'Hypothesis': 'H4 Medication change',
    'Variable': var,
    'Test': 'Chi-square test of independence',
    'Statistic': chi2,
    'Degrees_of_freedom': dof,
    'p_value': p_value,
    'Effect_size': cramers_v,
    'Effect_type': "Cramer's V",
    'Effect_magnitude': effect_label(cramers_v, 'cramers_v'),
    'Cohen_h_Ch_vs_No': cohen_h,
    'Cohen_h_magnitude': effect_label(cohen_h, 'cohen_h'),
    'Expected_cells_lt5': int(expected_lt5),
    'Decision': 'Reject H0' if p_value < 0.05 else 'Fail to reject H0'
}
pd.DataFrame([h4_result]).round(4)

Contingency table:


readmittedbinary,0,1
change,,
Ch,40577,5545
No,47452,5769


Cells with expected count < 5: 0


,Hypothesis,Variable,Test,Statistic,Degrees_of_freedom,p_value,Effect_size,Effect_type,Effect_magnitude,Cohen_h_Ch_vs_No,Cohen_h_magnitude,Expected_cells_lt5,Decision
0,H4 Medication change,change,Chi-square test of independence,34.1342,1,0.0,0.0185,Cramer's V,negligible,0.0372,small,0,Reject H0


## Combined results table
This summary brings all four hypothesis tests together so the evidence can be compared in one place.


In [16]:
results = pd.DataFrame([h1_result, h2_result, h3_result, h4_result])
results_display = results[[
    'Hypothesis', 'Variable', 'Test', 'Statistic', 'p_value',
    'Effect_type', 'Effect_size', 'Effect_magnitude', 'Decision'
]].copy()
results_display = results_display.sort_values('p_value')
results_display.round(4)


,Hypothesis,Variable,Test,Statistic,p_value,Effect_type,Effect_size,Effect_magnitude,Decision
0,H1 Prior inpatient utilisation,numberinpatient,Mann-Whitney U,6.048153e+08,0.0,Cliff's delta,0.2145,small,Reject H0
2,H3 Discharge disposition,dischargedisposition,Chi-square test of independence,1.225264e+03,0.0,Cramer's V,0.1111,small,Reject H0
1,H2 Length of stay,timeinhospital,Mann-Whitney U,5.454889e+08,0.0,Cliff's delta,0.0954,negligible,Reject H0
3,H4 Medication change,change,Chi-square test of independence,3.413420e+01,0.0,Cramer's V,0.0185,negligible,Reject H0


## Interpretation notes
- Statistical significance should be interpreted together with effect size. In large healthcare datasets, tiny differences often become statistically significant.
- If H1 is supported, prior inpatient utilisation should be treated as a high-priority modelling feature.
- If H2 is significant but effect size is weak, length of stay may still be useful in a model, but probably as a secondary predictor.
- If H3 is significant, discharge disposition is likely one of the strongest operational risk indicators.
- If H4 is significant but small in magnitude, medication change may reflect clinical complexity more than direct causation.


## Next notebook-ready extension ideas
After this notebook, the project can naturally move into:
- diagnosis-category hypothesis testing,
- insulin and diabetes medication subgroup testing,
- logistic regression with odds ratios,
- feature engineering for modelling,
- train/test modelling notebook.
